In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv

def _find_upwards(filename: str, start: Path) -> Path | None:
    for p in [start, *start.parents]:
        candidate = p / filename
        if candidate.exists():
            return candidate
    return None

env_file = _find_upwards(".env.local", Path.cwd())
if env_file is None:
    print("Warning: .env.local not found from current working directory upward.")
else:
    load_dotenv(env_file, override=False)
    print(f"Loaded env file: {env_file}")

print("GOOGLE_PLACES_API_KEY present:", bool(os.environ.get("GOOGLE_PLACES_API_KEY")))

Loaded env file: c:\Users\kylec\OneDrive\Desktop\react_project\london-explorer\.env.local
GOOGLE_PLACES_API_KEY present: True


In [2]:
from h3_loadboundary import load_boundary_from_json
from h3_seedtiles import init_h3
# import sys
# from pathlib import Path
# HERE = Path.cwd()
# PARENT = HERE.parent  # server/scripts
# if str(PARENT) not in sys.path:
#     sys.path.insert(0, str(PARENT))
# from request_places.request_places import nearby_search
# # or: from request_places.request_places import request_places

borough_geodf, inner_union = load_boundary_from_json()
seed_geodf = init_h3(inner_union)

In [3]:
from h3_selecttiles import select_tiles_in_radius
from h3_visualizemap import visualize_seed
# seed_geodf = select_tiles_in_radius(seed_geodf, 51.456644, -0.143151, 800)
seed_geodf = select_tiles_in_radius(seed_geodf, 51.469843, -0.138641, 800)
seed_geodf.to_csv("out/seed_tiles.csv", index=False)  # Debug output
visualize_seed(seed_geodf, inner_union)

Saved seed H3 map to: out/h3hex_seed_map.html


In [4]:
# import importlib
# import h3_subdivision
# importlib.reload(h3_subdivision)
from h3_subdivision import run_h3_recursive_division
DISABLE_API = True
division_geodf = await run_h3_recursive_division(seed_geodf, inner_union, disable_api=DISABLE_API)
if not DISABLE_API: division_geodf.to_csv("out/division_tiles.csv", index=False)  # Debug output

API calls executed: 1 | failures: 0
API calls executed: 2 | failures: 0
API calls executed: 3 | failures: 0
API calls executed: 4 | failures: 0
API calls executed: 5 | failures: 0
API calls executed: 6 | failures: 0
API calls executed: 7 | failures: 0
API calls executed: 8 | failures: 0
API calls executed: 9 | failures: 0
API calls executed: 10 | failures: 0
API calls executed: 11 | failures: 0
API calls executed: 12 | failures: 0
API calls executed: 13 | failures: 0
API calls executed: 14 | failures: 0
API calls executed: 15 | failures: 0
API calls executed: 16 | failures: 0
API calls executed: 17 | failures: 0
API calls executed: 18 | failures: 0
API calls executed: 19 | failures: 0
API calls executed: 20 | failures: 0
API calls executed: 21 | failures: 0
API calls executed: 22 | failures: 0
API calls executed: 23 | failures: 0
API calls executed: 24 | failures: 0
API calls executed: 25 | failures: 0
API calls executed: 26 | failures: 0
API calls executed: 27 | failures: 0
API calls 

In [5]:
from h3_visualizemap import visualize_divisions
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely import wkt
from config import SOURCE_CRS

division_csv = Path("out/division_tiles.csv")
if not division_csv.exists():
    raise FileNotFoundError("out/division_tiles.csv not found. Run the division step first.")
df = pd.read_csv(division_csv)
if "geometry" not in df.columns:
    raise ValueError("division_tiles.csv is missing geometry, cannot plot polygons.")
df["geometry"] = df["geometry"].apply(wkt.loads)
division_geodf_READ = gpd.GeoDataFrame(df, geometry="geometry", crs=SOURCE_CRS)

visualize_divisions(division_geodf_READ, inner_union)

Saved mock adaptive H3 map to: out/h3hex_mock_map.html
